In [1]:
!pip install pandas
!pip install vaderSentiment

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
  Using cached vaderSentiment-3.3.2-py2.py3-none-any.whl.metadata (572 bytes)
Using cached vaderSentiment-3.3.2-py2.py3-none-any.whl (125 kB)



[notice] A new release of pip is available: 24.3.1 -> 25.1.1
[notice] To update, run: C:\Users\User\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import re
import pickle
import numpy as np
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [3]:
# Load vectorizer
vectorizer = pickle.load(open("../models/vectorizer.sav", "rb"))

# Load model
nb_model = pickle.load(open("../models/trad_model.sav", "rb"))
lr_model = pickle.load(open("../models/lr_model.sav", "rb"))

In [4]:
def handle_negations(s: str):
    negation_pattern = r'\b(not|no|never|none|cannot|cant|wont|dont)\b[\w\s]+'
    return re.sub(negation_pattern, lambda match: match.group(0).replace(' ', '_'), s)

""" Removes unnecessary symbols from the text """
def clean_text(s: str):
    # Only retain alphanumeric and whitespace characters
    s = re.sub(pattern=rf"[^a-zA-Z0-9\s]", repl="", string=s, flags=re.IGNORECASE)

    # Convert to lowercase
    s = s.lower()

    # Remove extra whitespaces
    s = re.sub(pattern=r"\s+", repl=" ", string=s).strip()

    return s

""" Implements pipeline of pre-processing techniques """
def preprocess(text: str):
    return handle_negations(clean_text(text))

""" Sentiment Scorer """
def sentiment_scores(sentence):
    sid_obj = SentimentIntensityAnalyzer()
    sentiment_dict = sid_obj.polarity_scores(sentence)

    return sentiment_dict['compound']

In [5]:
""" Return predictions and probability """
def predict(x, sentiment_scores):
    print(np.array(x.toarray()).shape)
    # Predict on the two models
    mnb_pred = nb_model.predict_proba(x)
    lr_pred = lr_model.predict_proba(sentiment_scores.to_numpy().reshape(-1,1))

    # Model weights
    mnb_w, lr_w = 0.4, 0.6

    tot_pred = mnb_pred * mnb_w + lr_pred * lr_w

    # Get the index of the highest probability
    # return np.argmax(mnb_pred, axis=1), [max(prob) for prob in mnb_pred]
    return np.argmax(tot_pred, axis=1), [max(prob) for prob in tot_pred]

In [13]:
# New data to test the model
# with open('../data/demo_input.txt', 'r') as f:
#     new_data = f.read().split('\n')

#     df_test = pd.DataFrame({'text' : new_data})
df_test = pd.read_csv('../data/demo.csv')
new_data = df_test['text']

df_test["cleaned"] = df_test["text"].apply(preprocess)
df_test["sentiment_score"] = df_test["text"].apply(sentiment_scores)

X_test = vectorizer.transform(df_test["cleaned"])

result, prob = predict(X_test, df_test["sentiment_score"])

# Map the predictions to labels (if applicable)
class_names = {0: "Risk", 1: "Neutral", 2: "Opportunity"}

# Display predictions
for text, label, p in zip(new_data, result, prob):
    print(f"Text: {text}\nPredicted Label: {class_names[label]}\nConfidence: {p}\n")

(6, 1600)
Text: With reference to the reuse of materials, the projects created in implementation of the Group’s investment plans for the Italian motorway network provided for the reuse - within the regulatory limits - of the earth deriving from excavations, in order to mitigate the environmental impact linked mainly to the procurement of inert quarry materials and the disposal in landfills of unused materials. They are reused to create embankments, landscaping and noise-absorbing dunes, as well as for the redevelopment of degraded areas (such as abandoned quarries).
Predicted Label: Risk
Confidence: 0.45533062481289205

Text: 2019 also saw the continuation of activities regarding lighting, with widespread use of LED technology, both in motorway tunnels and airports, as well as in buildings, which reduced electrical energy consumption by around 5.4 GWh. As regards air conditioning, modernisation of the systems continued, with more efficient machinery, such as refrigeration units, signif